# Segmento 3: RAG, Retrieval-Augmented Generation

Abbiamo un vector database che trova le ricette più rilevanti per qualsiasi query.

Ora colleghiamolo a un LLM: prima **recuperiamo** i dati, poi **generiamo** una risposta basata su dati reali.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
import json, os

load_dotenv()
openai_client = OpenAI()
qdrant = QdrantClient(host="localhost", port=6333)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-5.4-nano"
COLLECTION = "recipes"

# Verifichiamo che Qdrant abbia i nostri dati
info = qdrant.get_collection(COLLECTION)
print(f"Collection '{COLLECTION}': {info.points_count} ricette indicizzate")

Collection 'recipes': 1000 ricette indicizzate


## Step 1: La funzione di retrieval

Una semplice funzione che prende una query e restituisce le 5 ricette più rilevanti da Qdrant.

In [2]:
def retrieve_recipes(query, top_n=5):
    """Retrieve the most relevant recipes for a given query."""
    # Step 1: Embeddiamo la query
    response = openai_client.embeddings.create(input=[query], model=EMBED_MODEL)
    query_vector = response.data[0].embedding
    
    # Step 2: Cerchiamo in Qdrant
    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        limit=top_n,
        with_payload=True,
    )
    
    # Step 3: Formattiamo i risultati
    recipes = []
    for point in results.points:
        recipes.append({
            "title": point.payload["title"],
            "ingredients": point.payload["ingredients"],
            "instructions": point.payload["instructions"],
            "score": round(point.score, 4),
        })
    return recipes

# Proviamo
results = retrieve_recipes("pasta with cheese")
for r in results:
    print(f"  {r['score']:.4f}  {r['title']}")

  0.5815  Cheesy Baked Pasta with Cauliflower
  0.5588  Spaghetti with Pecorino Romano and Black Pepper
  0.5367  Antipasto Pasta
  0.5110  Penne with Tomato Pesto and Smoked Mozzarella
  0.5060  Wagon-Wheel Pasta & Goat Cheese


## Step 2: Costruire il prompt RAG

Prendiamo la domanda dell'utente + le ricette recuperate e costruiamo un prompt per l'LLM.

L'istruzione chiave: **rispondi usando solo le ricette fornite**, niente allucinazioni.

In [3]:
RAG_SYSTEM_PROMPT = """You are a helpful cooking assistant. You answer questions about recipes.

You will be given a set of recipes retrieved from a database. Use ONLY these recipes to answer.
If the retrieved recipes don't contain enough information to answer, say so honestly.

When suggesting a recipe, mention its name and briefly describe why it fits the user's request.
Keep your answers concise and practical."""

def build_rag_prompt(question, recipes):
    """Build the messages list for the RAG call."""
    # Formattiamo le ricette recuperate come contesto
    context = "\n\n---\n\n".join([
        f"**{r['title']}** (relevance: {r['score']})\n"
        f"Ingredients: {r['ingredients']}\n"
        f"Instructions: {r['instructions']}"
        for r in recipes
    ])
    
    user_message = f"""Here are the most relevant recipes from the database:

{context}

---

User question: {question}"""
    
    return [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

# Anteprima del prompt
question = "What can I cook with leftover chicken?"
recipes = retrieve_recipes(question)
messages = build_rag_prompt(question, recipes)

print("System prompt:")
print(messages[0]["content"])
print("\n" + "="*60 + "\n")
print("User message (primi 800 caratteri):")
print(messages[1]["content"][:800])

System prompt:
You are a helpful cooking assistant. You answer questions about recipes.

You will be given a set of recipes retrieved from a database. Use ONLY these recipes to answer.
If the retrieved recipes don't contain enough information to answer, say so honestly.

When suggesting a recipe, mention its name and briefly describe why it fits the user's request.
Keep your answers concise and practical.


User message (primi 800 caratteri):
Here are the most relevant recipes from the database:

**Chicken and Ginger Soup** (relevance: 0.4819)
Ingredients: ['12 long fresh cilantro stems', '6 garlic cloves, peeled', '1 2-inch piece peeled fresh ginger, chopped, plus matchstick-size strips for garnish', '10 white peppercorns', '3 tablespoons vegetable oil', '1/2 cup Chinese rice wine', '6 tablespoons oyster sauce', '6 tablespoons yellow bean sauce', '1/4 cup sugar', '8 cups low-salt chicken broth', '4 large organic or free-range chicken thighs with skin and bones, rinsed', '4 large organ

## Step 3: Chiedere all'LLM

Ora mettiamo tutto insieme: **retrieve → costruisci prompt → genera risposta**.

In [4]:
def ask(question):
    """Full RAG pipeline: retrieve recipes, then ask the LLM."""
    # 1. Recupera
    recipes = retrieve_recipes(question)
    print(f"Recuperate {len(recipes)} ricette:")
    for r in recipes:
        print(f"  {r['score']:.4f}  {r['title']}")
    print()
    
    # 2. Costruisci il prompt
    messages = build_rag_prompt(question, recipes)
    
    # 3. Genera la risposta
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
    )
    
    answer = response.choices[0].message.content
    print(f"Risposta:\n{answer}")
    return answer

ask("What can I cook with leftover chicken?");

Recuperate 5 ricette:
  0.4819  Chicken and Ginger Soup
  0.4784  Colombian Chicken, Corn, and Potato Stew
  0.4743  Chicken Stock on the Grill
  0.4737  Chicken Taquitos
  0.4732  Barbecued Chicken and Chickpea Quesadillas



Risposta:
Here are a few good options from the recipes you have that work well for leftover chicken:

- **Chicken Taquitos** — Use the leftover chicken as the filling (it’s shredded and mixed with BBQ sauce, sour cream, chiles, corn, cheese, and spices, then rolled and fried/pan-fried).
- **Barbecued Chicken and Chickpea Quesadillas** — Mix leftover shredded chicken with BBQ sauce, then pair it with a mashed chickpea mixture and cheese inside tortillas.
- **Chicken and Ginger Soup** — Shred or chop leftover chicken and stir it into this flavorful ginger/garlic/Chinese rice wine soup near the end (the recipe is designed around chicken pieces simmered until tender, so leftover chicken just needs to be warmed through).
- **Colombian Chicken, Corn, and Potato Stew** — If you have enough leftover chicken, you can use it in the stew for a hearty meal with potatoes and corn (this recipe simmers chicken to cook through, so you’d adjust timing since your chicken is already cooked).

If you tell

In [5]:
ask("I want something light and fresh for summer");

Recuperate 5 ricette:
  0.3813  Fresh Berries with Ricotta Cream
  0.3460  Easy Does It, Baby
  0.3344  Simple Spring Green Salad
  0.3331  Summer Salmon Cakes with Zucchini Fennel Slaw
  0.3326  Watermelon with Fennel Salt



Risposta:
For something light and fresh in summer, I’d suggest:

**Simple Spring Green Salad** — It’s very light and refreshing: just torn mixed lettuces, a little olive oil (plus salt), and serve immediately (no heavy sauce or cooking).

If you want something a bit more “summer” (fruit-forward):

**Watermelon with Fennel Salt** — Chilled watermelon with toasted fennel-salt and lime for a clean, refreshing flavor.


In [6]:
ask("Something warm and comforting, I'm feeling lazy");

Recuperate 5 ricette:
  0.2984  Warm-Spiced Saucy Lamb Stew
  0.2804  Winter Minestrone
  0.2802  Ham Hock and White Bean Stew
  0.2735  Chilled Tomato and Stone Fruit Soup
  0.2703  Hot Toddy



Risposta:
**Warm-Spiced Saucy Lamb Stew** — warm, hearty, and “saucy” comfort food. It’s a big one-pot-style stew and you can lean on lazy shortcuts like using pre-chopped onion/garlic and skipping the optional serving garnishes.

**Winter Minestrone** — also very comforting and filling, with lots of vegetables and beans. If you want the laziest approach, it’s easiest to treat it like a “dump-and-simmer” soup (prep is mostly chopping, then long cooking).

**Ham Hock and White Bean Stew** — rich and cozy, and it’s great if you like hands-off cooking: once it’s going, you can just let it simmer while you do nothing.

If you tell me **meat vs. vegetarian** and whether you have **a pot/Dutch oven**, I’ll point you to the single best match.


In [7]:
# Una query che mostra il retrieval per intento, non per keyword
ask("Mi suggerisci un piatto veloce per cena per due persone");

Recuperate 5 ricette:
  0.4680  Seafood Stew for Two
  0.4184  Rigatoni with Shrimp, Calamari and Basil
  0.4082  Mozzarella Arrabiata Salsa
  0.4044  Soft Polenta with Mushrooms and Spinach
  0.3976  Spaghetti with Pecorino Romano and Black Pepper



Risposta:
Certo! Dato che vuoi **una cena veloce per due persone**, ti consiglio:

### **Spaghetti con Pecorino Romano e Pepe Nero**
È un piatto molto semplice e rapido: si fa **tostando/crushando il pepe**, cuocendo gli **spaghetti al dente**, e poi mantecando/condendo con **Pecorino Romano**. Perfetto se hai poco tempo ma vuoi qualcosa di gustoso.

Se mi dici anche cosa hai in casa (carne/pesce/verdure e che tipo di pasta hai), posso suggerirti un’alternativa ancora più adatta.


## Cosa abbiamo costruito

Una **pipeline RAG**:

1. L'utente fa una domanda in linguaggio naturale
2. **Embeddiamo** la domanda e **cerchiamo** nel vector database
3. Le ricette migliori diventano il **contesto** per l'LLM
4. L'LLM genera una risposta **basata su dati reali**

L'LLM non si inventa le ricette, lavora con quello che il database gli fornisce.

**Tra poco**: e se l'LLM potesse **decidere da solo** quando cercare? Quello è il function calling, il primo passo verso gli agenti.